# Solution: Graph Neural Networks with PyTorch Geometric

GNNs are a class of neural networks that can work with graph data. 

GNNs propagate nodal information throughout the graph via graph shifting.
Generally, we define as input a node feature matrix $\mathbf{X} \in \mathbb{R}^{N \times F}$, with $F$ the number of node features, which represents signals/values for each node in the graph.
The layer propagation rule is defined as follows:

$\textbf{Y} = \sum_{k=0}^K \mathbf{S}^k \mathbf{X} \mathbf{H}_k$

where $\mathbf{Y} \in \mathbb{R}^{N \times G}$ is the output node feature matrix, $K$ is the K-hop neighbourhood and represent how many times the node information propagates across its neighbouring nodes, $\mathbf{S} \in \mathbb{R}^{N \times N}$ is a graph shift operator, and $\mathbb{H}=\{\mathbf{H}_k : k=0,1,...,K\} \in \mathbb{R}^{K \times F × G}$ is a set of learnable matrices.

<!-- <center><figure>

<figcaption>Figure 1. A graph neural network (GNN) layer with a 2-hop neighbourhood. 

<sub><sup> The figures from left to right indicate how the node signal in the black node propagates throughout the network. The same reasoning is applied to every other node in the graph.  </sup></sub></figcaption>

</figure></center> -->

In [ ]:
# Libraries
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch_geometric as pyg
from sklearn.model_selection import train_test_split
from torch_geometric.loader import DataLoader
from torch_geometric.datasets import TUDataset
from torch_geometric.nn import GCNConv, MessagePassing
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix
torch.manual_seed(42);

# Dataset

In this notebook, we'll use the MUTAG dataset.
The MUTAG dataset in PyTorch Geometric is a benchmark dataset for graph classification tasks. The MUTAG dataset is used for predicting mutagenicity of chemical compounds. Here the task is a binary graph-level classification tasks.

Here's a brief description of the MUTAG dataset:

- Task: Graph Classification (predict whether a chemical compound is mutagenic or non-mutagenic)
- Graph Structure: each graph in the dataset represents a chemical compound, in which nodes represent atoms, and edges represent chemical bonds between atoms.
- Node Features: represent atom types, and they encode information about the type of atoms in the chemical compound.
- Edge Features: one-hot-encoded vector defining the type of chemical bond.
- Graph Labels: each graph is labeled as either mutagenic (1) or non-mutagenic (0).

In [ ]:
dataset = TUDataset(root='../../../../../datasets/MUTAG', name='MUTAG', use_node_attr=True, use_edge_attr=True)

In [ ]:
# Print information about the dataset
print(f'Dataset: {dataset}:')
print(f'Number of graphs: {len(dataset)}')
print(f'Number of node features: {dataset.num_node_features}')
print(f'Number of edge features: {dataset.num_edge_features}')
print(f'Number of classes: {dataset.num_classes}')

data = dataset[0]  # Get the first graph object.
print(data)

## Create training and testing datasets

In [ ]:
dataset.shuffle()

train_dataset, test_dataset = train_test_split(dataset, test_size=0.2, random_state=42)
train_dataset, val_dataset = train_test_split(train_dataset, test_size=0.25, random_state=42)

In [ ]:
len(train_dataset), len(val_dataset), len(test_dataset)

# Model creation
## Message Passing

For small graphs, multiplying by the adjacency matrix, i.e., $\mathbf{A}\mathbf{X}$, is feasible. However, it becomes computationally prohibitive for large graphs as the computations are $\mathcal{O}(N^2)$. 

For this reason, PyG uses the sparse formulation with the edge indices and the local message-passing framework, which you saw in the advanced GNN architectures.
This changes the formulation from a global graph-wise perspective to a local node-wise one. 
In particular, each message passing GNN layer can only consider 1-hop neighbourhoods. So, to cover larger portions of the graph you have to stack multiple layers.

The message-passing GNN is expressed as:

- Edge update:

$\mathbf{e}_{j,i}^{(k)} = \phi^{(k)}\left(\mathbf{x}_i^{(k-1)}, \mathbf{x}_j^{(k-1)},\mathbf{e}_{j,i}^{(k-1)}\right)$,

- Node update:

$\mathbf{x}_i^{(k)} = \gamma^{(k)} \left( \mathbf{x}_i^{(k-1)}, \square_{j \in \mathcal{N}(i)} \, \mathbf{e}_{j,i}^{(k)} \right)$,

where $(k)$ represents the $k^{th}$ layer in the network, $\square$ denotes a differentiable, permutation invariant function (e.g., sum, mean or max) and $\gamma$ and $\phi$ denote differentiable functions such as MLPs (Multi Layer Perceptrons).

With PyG you can easily create your own GNN class by changing the edge update equation (in the *message* method) and the node update equation (in *forward*).

Below you can see an example of how to create a 1-layer graph convolutional neural network with the following edge and node equations:

- Edge update:

$\mathbf{e}_{j,i}^{(k)} = A_{j,i} \mathbf{x}_i^{(k-1)}$

- Node update:

$\mathbf{x}_i^{(k)} = \sum_{j \in \mathcal{N}(i)} \mathbf{e}_{j,i}^{(k)} \mathbf{H}$

This is equivalent to a standard GNN with 1-hop neighbourhood, defined as $\textbf{Y} = \mathbf{A} \mathbf{X} \mathbf{H}$.

In [ ]:
from torch_geometric.typing import OptTensor

class one_hop_GNN(MessagePassing):
    def __init__(self, in_features, out_features):
        super().__init__(aggr='add')  # "Add" aggregation (Step 3).
        torch.manual_seed(42)
        self.lin = nn.Linear(in_features, out_features, bias=False)

    def forward(self, x, edge_index, edge_weight):
        # The inputs of forward are the node feature matrix (x), 
        # the graph connectivity (edge_index), and the edge weight (the adjacency values)

        # Step 1: Linearly transform node feature matrix (X H).
        x = self.lin(x)

        # Step 2-3: Start propagating messages (A (X H)).
        out = self.propagate(edge_index, x=x, edge_weight=edge_weight)

        return out

    def message(self, x_j, edge_weight: OptTensor):
        # Here we create e_{ji} by multiplying the edge weight (A_{ij}) with the node feature of the source node
        # If the adjacency matrix is unweighted, then A_{ij} = 1 so we can simply return x_j
        return x_j if edge_weight is None else edge_weight.view(-1, 1) * x_j
    
in_features = dataset.num_node_features
out_features = dataset.num_classes

model = one_hop_GNN(in_features, out_features)

y = model(data.x, data.edge_index, data.edge_weight)

print(y.shape)

Luckily, we don't need to manually write every GNN we want to implement as PyG provides a huge variety of pre-implemented models (https://pytorch-geometric.readthedocs.io/en/latest/cheatsheet/gnn_cheatsheet.html).
In the following, we will only use the pre-defined model given by PyG.

As you might know by now, GNN represent just another type of neural network layer, so we can create an architecture with multiple layers, as shown below.

In [ ]:
class GCN(torch.nn.Module):
    def __init__(self, in_features, out_features, hidden_features):
        super().__init__()
        torch.manual_seed(42)
        # GCNConv is a graph convolutional layer, as we implemented before (but with more options)
        self.conv1 = GCNConv(in_features, hidden_features)
        self.conv2 = GCNConv(hidden_features, hidden_features)
        self.dropout = nn.Dropout(0.1)
        self.fc = nn.Linear(hidden_features, out_features)

    def forward(self, data):
        """data is the pyg object that contains (among the rest):
            - x: node feature matrix
            - edge_index: graph connectivity
            - batch_idx: for each node, the index of the graph it belongs to
        """
        x, edge_index, batch_idx = data.x, data.edge_index, data.batch

        x = self.conv1(x, edge_index)
        x = nn.ReLU()(x)
        x = self.conv2(x, edge_index)
        x = pyg.nn.global_mean_pool(x, batch_idx) # Average pooling
        x = self.dropout(x)
        x = self.fc(x)

        return x

model = GCN(in_features, out_features, hidden_features=128)
print(model)

**Exercise:**

In this exercise, you will implement the a Graph Attention Network (GAT), following the same procedure as before.
One advantage of using GAT is that it allows to explot as well edge features! Try adding that information as well in the forward pass.

Hint: the syntax is very similar to that of edge_index (you should also check the PyG documentation)

In [ ]:
class GAT(torch.nn.Module):
    def __init__(self, in_features, out_features, hidden_features):
        super().__init__()
        torch.manual_seed(42)
        # Add at least 2 GAT layers (they are called GATConv in PyG)
        # ---------------------- student exercise --------------------------------- #
        self.conv1 = pyg.nn.GATConv(in_features, hidden_features, heads=1)
        self.conv2 = pyg.nn.GATConv(hidden_features, hidden_features, heads=1)
        # ---------------------- student exercise --------------------------------- #
        self.dropout = nn.Dropout(0.1)
        self.fc = nn.Linear(hidden_features, out_features)
        
    def forward(self, data):
        """data is the pyg object that contains (among the rest):
            - x: node feature matrix
            - edge_index: graph connectivity
            - batch_idx: for each node, the index of the graph it belongs to
        """
        x, edge_index, batch_idx = data.x, data.edge_index, data.batch

    def forward(self, data):
        # Implement the forward pass
        # ---------------------- student exercise --------------------------------- #
        x, edge_index, batch_idx, edge_attr = data.x, data.edge_index, data.batch, data.edge_attr

        x = self.conv1(x, edge_index, edge_attr)
        x = nn.ReLU()(x)
        x = self.conv2(x, edge_index, edge_attr)
        # ---------------------- student exercise --------------------------------- #
        
        x = pyg.nn.global_mean_pool(x, batch_idx) # Average pooling
        x = self.dropout(x)
        x = self.fc(x)
        
        return x

model = GAT(in_features, out_features, hidden_features=128)
print(model)

# Training and testing

In [ ]:
def train_epoch(model, loader, optimizer, loss_function, device='cpu'):
    model.to(device)
    model.train()
    losses = []
    correct_preds = 0

    for batch in loader:
        batch = batch.to(device)
        preds = model(batch) 
        
        loss = loss_function(preds, batch.y)
        correct_preds += (preds.argmax(dim=1) == batch.y).sum()
        
        losses.append(loss.cpu().detach())
        
        loss.backward()   # compute the gradients using backpropagation
        optimizer.step()  # update the weights with the optimizer
        optimizer.zero_grad(set_to_none=True)   # reset the computed gradients      

    accuracy = correct_preds/len(loader.dataset)*100
    
    return np.array(losses).mean(), accuracy.item()

def evaluation(model, loader, loss_function, device='cpu'):
    model.to(device)
    model.eval() # specifies that the model is in evaluation mode
    losses = []
    correct_preds = 0
    
    # Remove gradients computations since we are only evaluating and not training
    with torch.no_grad():
        for batch in loader:
            batch = batch.to(device)
            preds = model(batch)
            
            loss = loss_function(preds, batch.y)
            correct_preds += (preds.argmax(dim=1) == batch.y).sum()

            losses.append(loss.cpu())

    accuracy = correct_preds/len(loader.dataset)*100

    return np.array(losses).mean(), accuracy.item()

In [ ]:
# Set training parameters
learning_rate = 0.001
batch_size = 8
num_epochs = 50

# Create the training and validation dataloaders to "feed" data to the model in batches
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

# Create the optimizer to train the neural network via back-propagation
optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)

# Create loss function
loss_function = nn.CrossEntropyLoss()

# This line is used to select GPU to train, if available
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [ ]:
train_losses = []
train_accuracies = []
val_losses = []
val_accuracies = []

for epoch in range(1, num_epochs+1):
    # Model training
    train_loss, train_accuracy = train_epoch(model, train_loader, optimizer, loss_function, device=device)
    val_loss, val_accuracy = evaluation(model, test_loader, loss_function, device=device)

    train_losses.append(train_loss)
    train_accuracies.append(train_accuracy)

    val_losses.append(val_loss)
    val_accuracies.append(val_accuracy)
    
    # print loss every epoch
    if epoch%5 == 0:
        print("epoch:",epoch, "\t training accuracy:", np.round(train_accuracy,2), 
                              "\t validation accuracy:", np.round(val_accuracy,2))

# Results

As always, let's first check that our loss function is doing its job by decreasing with the epochs.

In [ ]:
# plot loss and accuracy curves
plt.figure(figsize=(12,4))

plt.subplot(121)
plt.plot(train_losses, label='train')
plt.plot(val_losses, label='validation')
plt.xlabel('epochs')
plt.ylabel('loss')
plt.legend()

plt.subplot(122)
plt.plot(train_accuracies, label='train')
plt.plot(val_accuracies, label='validation')
plt.xlabel('epochs')
plt.ylabel('accuracy')
plt.legend()
plt.show()

Let's now check the performance on the testing dataset

In [ ]:
# test dataset
test_loss, test_accuracy = evaluation(model, test_loader, loss_function, device=device)
print("Test accuracy:", np.round(test_accuracy,2))

# gather all predictions and ground truth labels
preds = []
labels = []
for batch in test_loader:
    preds.append(model(batch.to(device)).argmax(dim=1))
    labels.append(batch.y)
preds = torch.cat(preds).cpu().numpy()
labels = torch.cat(labels).cpu().numpy()

# compare with ground truth labels 
print("Predictions:", preds)
print("Ground truth: ", labels)

In [ ]:
# Since it is a classification problem, we can also print the confusion matrix with some explanations
print(f"Confusion matrix:\n{confusion_matrix(labels, preds)}")